# Hướng dẫn tạo AI Agent đơn giản với Semantic Kernel

Tìm hiểu cách xây dựng AI Agent với Semantic Kernel chỉ trong vài bước. Hướng dẫn này bao gồm các yếu tố cần thiết: tạo agent, thêm công cụ và quản lý các cuộc hội thoại.

Có ba thành phần cơ bản trong Semantic Kernel agent:

1. Agent Class: Tất cả các loại agent đều kế thừa từ lớp này. Các loại agent bao gồm ChatCompletionAgent (sử dụng API chat completion tiêu chuẩn), OpenAIAssistantAgent (tận dụng OpenAI Assistant API với các công cụ tích hợp sẵn), AzureAIAgent (tích hợp với các dịch vụ Azure AI cho các kịch bản doanh nghiệp), CopilotStudioAgent (kết nối với các quy trình làm việc của Microsoft Copilot Studio).
2. Agent Thread: Thành phần này xử lý cách lịch sử cuộc hội thoại và trạng thái được duy trì. Điều này quan trọng vì các agent cần ngữ cảnh từ các tin nhắn trước đó để đưa ra quyết định sáng suốt. Có hai cách tiếp cận:
    1. Service managed state: Một dịch vụ agent như Azure AI lưu trữ lịch sử cuộc hội thoại phía máy chủ và được truy cập qua thread ID.
    2. Application managed state: Ứng dụng của bạn duy trì toàn bộ lịch sử trò chuyện và chuyển nó cho agent trong mỗi lần gọi.
3. Agent orchestration: Framework cung cấp các mẫu được xây dựng sẵn để điều phối nhiều agent nhằm xử lý các quy trình làm việc phức tạp mà các agent đơn lẻ không thể quản lý hiệu quả.
    1. Sequential: Các agent thực thi lần lượt theo trình tự. Điều này giống như một quy trình xử lý tài liệu.
    2. Concurrent: Nhiều agent làm việc cùng lúc. Giống như xử lý yêu cầu của khách hàng (agent thanh toán và agent tài khoản làm việc song song).
    3. Handoff: Các agent chuyển quyền kiểm soát cho nhau dựa trên chuyên môn. Giống như phân loại dịch vụ khách hàng cho agent chuyên gia.
    4. Group chat: Các agent hợp tác trong một cuộc hội thoại chung. Giống như lập kế hoạch dự án với các chuyên gia miền.

Agents tận dụng hệ thống plugin của Semantic Kernel để truy cập các công cụ, cơ sở dữ liệu, v.v.

Các agent cũng có thể được định nghĩa bằng YAML.

## Thiết lập

Cài đặt các gói cần thiết và cấu hình môi trường của bạn:

In [ ]:
# Install packages
!pip install semantic-kernel python-dotenv

# Import everything we need
from semantic_kernel import Kernel
from semantic_kernel.agents import ChatCompletionAgent
from semantic_kernel.connectors.ai.open_ai import OpenAIChatCompletion
from semantic_kernel.functions import kernel_function

print("✅ Setup complete!")

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Check if API key is configured
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    print("⚠️  Please add OPENAI_API_KEY to your .env file!")
    print("   Create a .env file with: OPENAI_API_KEY=your_actual_key_here")
else:
    print("✅ API key configured")

## Bước 1: Tạo một Agent đơn giản

Hãy bắt đầu với những điều cơ bản - một agent có thể trò chuyện. Chúng ta chỉ đơn giản là cấp cho kernel (bộ điều phối của chúng ta) quyền truy cập vào dịch vụ trò chuyện, sử dụng endpoint Chat Completion API từ OpenAI. Chúng ta sẽ thêm nhiều công cụ hơn sau này.

In [ ]:
# Create service and kernel
chat_service = OpenAIChatCompletion(
    ai_model_id="gpt-4o-mini",
    api_key=OPENAI_API_KEY
)

kernel = Kernel()
kernel.add_service(chat_service)

# Create a basic agent
agent = ChatCompletionAgent(
    kernel=kernel,
    name="Assistant",
    instructions="You are a helpful and friendly assistant."
)

# Test it
response = await agent.get_response("Hello! What can you help me with?")
print(f"🤖 {agent.name}: {response.content}")

## Bước 2: Thêm Tools (Functions)

Bây giờ hãy cung cấp cho agent của chúng ta một số khả năng hữu ích. Chúng ta sẽ cung cấp cho nó một hàm thời tiết và một máy tính. Cả hai đều là các hàm mà chúng ta định nghĩa trong mã của mình và cung cấp chúng cho kernel dưới dạng tools.

In [ ]:
# Define useful functions
@kernel_function(description="Get current weather for a city")
def get_weather(city: str) -> str:
    """Mock weather function - replace with real API call."""
    weather_data = {
        "london": "Cloudy, 15°C",
        "paris": "Sunny, 22°C", 
        "tokyo": "Rainy, 18°C",
        "new york": "Partly cloudy, 20°C"
    }
    return weather_data.get(city.lower(), f"Weather data not available for {city}")

@kernel_function(description="Calculate simple math expressions")
def calculate(expression: str) -> str:
    """Safe calculator for basic math."""
    try:
        # Only allow basic math operations for safety
        allowed_chars = "0123456789+-*/(). "
        if all(c in allowed_chars for c in expression):
            result = eval(expression)
            return f"{expression} = {result}"
        else:
            return "Sorry, I can only do basic math operations (+, -, *, /, parentheses)"
    except:
        return "Sorry, I couldn't calculate that. Please check your expression."

# Add functions to kernel
kernel.add_function(plugin_name="tools", function=get_weather)
kernel.add_function(plugin_name="tools", function=calculate)

# Create enhanced agent
enhanced_agent = ChatCompletionAgent(
    kernel=kernel,
    name="SmartAssistant",
    instructions="""
    You are a helpful assistant with weather and calculator capabilities.
    
    - Use get_weather when asked about weather in specific cities
    - Use calculate for math problems
    - Be friendly and explain what you're doing
    """
)

print("✅ Enhanced agent created with tools!")

## Bước 3: Kiểm tra Agent với các công cụ

Hãy xem agent của chúng ta sử dụng các công cụ của nó:

In [ ]:
# Test weather function
print("🌤️ Testing weather function:")
response = await enhanced_agent.get_response("What's the weather in London?")
print(f"🤖 {enhanced_agent.name}: {response.content}\n")

# Test calculator function  
print("🧮 Testing calculator function:")
response = await enhanced_agent.get_response("What's 25 * 4 + 10?")
print(f"🤖 {enhanced_agent.name}: {response.content}\n")

# Test general conversation
print("💬 Testing general conversation:")
response = await enhanced_agent.get_response("Tell me a fun fact about AI")
print(f"🤖 {enhanced_agent.name}: {response.content}")

## Bước 4: Đối thoại có Bộ nhớ

Đối với các cuộc đối thoại ghi nhớ các tin nhắn trước đó. Chúng tôi sẽ sử dụng quản lý bộ nhớ có sẵn của Semantic Kernel, nhưng điều đó sẽ phụ thuộc vào cửa sổ ngữ cảnh.

Bộ nhớ có thể được giải quyết bằng cách tóm tắt các cuộc trò chuyện trước đây hoặc sử dụng bộ nhớ dài hạn hơn như RAG. Ngoài ra còn có bộ nhớ học tập nơi chúng tôi muốn AI Agent học từ tất cả các tương tác trong quá khứ, điều này cũng được triển khai bằng RAG, nơi chúng tôi lưu trữ các ví dụ giải quyết thành công và truy xuất chúng cho các trường hợp tương tự.

In [ ]:
async def chat_with_memory():
    """Demonstrate conversation with memory."""
    
    print("💭 Conversation with Memory Demo")
    print("=" * 40)
    
    # Messages that build on each other
    messages = [
        "Hi! I'm planning a trip to Paris.",
        "What's the weather like there?",
        "That sounds nice! Can you calculate 150 * 7 for my budget?",
        "Perfect, that should cover my week there. Thanks!"
    ]
    
    thread = None  # This will store conversation history
    
    for msg in messages:
        print(f"👤 User: {msg}")
        
        # Agent remembers previous messages through the thread
        response = await enhanced_agent.get_response(messages=msg, thread=thread)
        print(f"🤖 {enhanced_agent.name}: {response.content}\n")
        
        # Update thread to keep conversation history
        thread = response.thread

# Run the conversation demo
await chat_with_memory()

## Bước 5: Xem các Tool hoạt động (Nâng cao)

Hãy xem chính xác điều gì xảy ra khi AI Agent của bạn sử dụng các tool:

In [ ]:
async def show_tool_usage():
    """Show detailed tool execution."""
    
    print("🔧 Tool Usage Demo")
    print("=" * 25)
    
    # Callback to see tool calls
    async def log_tool_calls(message):
        from semantic_kernel.contents import FunctionCallContent, FunctionResultContent
        
        for item in message.items or []:
            if isinstance(item, FunctionCallContent):
                print(f"  🔧 Calling: {item.name}({item.arguments})")
            elif isinstance(item, FunctionResultContent):
                print(f"  ✅ Result: {item.result}")
    
    user_input = "What's the weather in Tokyo and what's 15 + 27?"
    print(f"👤 User: {user_input}\n")
    
    # Use invoke to see intermediate steps
    async for response in enhanced_agent.invoke(
        messages=user_input,
        on_intermediate_message=log_tool_calls
    ):
        print(f"\n🤖 Final Response: {response.content}")

# Run the tool usage demo
await show_tool_usage()

## Bước 6: Phản hồi Dạng Streaming

Đối với các phản hồi dạng thời gian thực (như ChatGPT):

In [ ]:
async def streaming_demo():
    """Show streaming responses."""
    
    print("\n🌊 Streaming Demo")
    print("=" * 20)
    
    print("👤 User: Write a short poem about coding")
    print("🤖 Assistant: ", end="", flush=True)
    
    # Stream response word by word
    async for chunk in enhanced_agent.invoke_stream(
        messages="Write a short poem about coding"
    ):
        print(chunk.content, end="", flush=True)
    
    print("\n")  # New line when done

# Run streaming demo
await streaming_demo()

## Tóm tắt: Những gì bạn đã học

In [ ]:
print("🎓 What You've Built:")
print("=" * 30)

summary = [
    "✅ Basic AI agent with OpenAI",
    "✅ Custom tools/functions for weather and math", 
    "✅ Conversation memory management",
    "✅ Tool execution monitoring",
    "✅ Real-time streaming responses"
]

for item in summary:
    print(item)

print("\n🚀 Key Concepts:")
concepts = {
    "Agent": "AI that can reason, remember, and use tools",
    "Kernel": "Manages AI services and functions",
    "Functions": "Tools that extend agent capabilities", 
    "Thread": "Maintains conversation history",
    "Streaming": "Real-time response generation"
}

for concept, description in concepts.items():
    print(f"• {concept}: {description}")

print("\n💡 Next Steps:")
print("• Try different models (gpt-4, gpt-3.5-turbo)")
print("• Create custom functions for your use case")
print("• Explore multi-agent conversations")
print("• Add guardrails for production safety")

## Bước 7: Điều phối tác nhân tuần tự

Bây giờ chúng ta hãy xem cách nhiều tác nhân có thể làm việc cùng nhau trong một pipeline - mỗi tác nhân xử lý đầu ra từ tác nhân trước đó. Hãy chú ý Semantic Kernel giúp chúng ta việc này dễ dàng như thế nào:

In [ ]:
# Import orchestration components
from semantic_kernel.agents import SequentialOrchestration
from semantic_kernel.agents.runtime import InProcessRuntime
from semantic_kernel.contents import ChatMessageContent

print("🔗 Setting up Sequential Agent Pipeline")
print("=" * 45)

In [ ]:
# Create specialized agents for a marketing pipeline
def create_marketing_pipeline():
    """Create three agents that work together sequentially."""
    
    # Agent 1: Extract key information
    concept_extractor = ChatCompletionAgent(
        name="ConceptExtractor",
        instructions="""
        You are a marketing analyst. Given a product description, identify:
        - Key features (bullet points)
        - Target audience 
        - Unique selling points
        
        Format your output clearly with headers.
        """,
        kernel=kernel
    )
    
    # Agent 2: Write marketing copy
    copywriter = ChatCompletionAgent(
        name="Copywriter", 
        instructions="""
        You are a marketing copywriter. Take the analysis provided and write 
        compelling marketing copy (around 100-150 words). Make it engaging 
        and highlight the key benefits. Output just the marketing copy.
        """,
        kernel=kernel
    )
    
    # Agent 3: Polish and format
    editor = ChatCompletionAgent(
        name="Editor",
        instructions="""
        You are an editor. Take the marketing copy and polish it:
        - Fix grammar and clarity
        - Ensure consistent tone
        - Make it more compelling
        - Output the final polished version
        """,
        kernel=kernel
    )
    
    return [concept_extractor, copywriter, editor]

# Create the pipeline
marketing_agents = create_marketing_pipeline()
print(f"✅ Created {len(marketing_agents)} specialized agents:")
for agent in marketing_agents:
    print(f"   • {agent.name}")

In [ ]:
# Set up callback to see each agent's work
def agent_callback(message: ChatMessageContent) -> None:
    """Show what each agent produces."""
    print(f"\n🤖 {message.name}:")
    print("-" * 30)
    print(message.content)
    print()

# Create the sequential orchestration
sequential_pipeline = SequentialOrchestration(
    members=marketing_agents,
    agent_response_callback=agent_callback
)

print("🔗 Sequential pipeline created!")

In [ ]:
# Run the sequential pipeline
async def run_marketing_pipeline():
    """Execute the sequential agent pipeline."""
    
    print("🚀 Running Marketing Pipeline")
    print("=" * 35)
    
    # Start the runtime
    runtime = InProcessRuntime()
    runtime.start()
    
    try:
        # Input: Product description
        product_description = (
            "A smart water bottle with temperature display, "
            "app connectivity, hydration reminders, and "
            "leak-proof design. Made from BPA-free materials."
        )
        
        print(f"📝 Input Product: {product_description}\n")
        print("Processing through pipeline...")
        
        # Run the sequential orchestration
        result = await sequential_pipeline.invoke(
            task=product_description,
            runtime=runtime
        )
        
        # Get final result
        final_output = await result.get(timeout=30)
        
        print("🎯 FINAL MARKETING COPY:")
        print("=" * 40)
        print(final_output)
        
    finally:
        # Clean up
        await runtime.stop_when_idle()

# Execute the pipeline
await run_marketing_pipeline()

In [ ]:
# Quick demo with a different product
async def quick_pipeline_demo():
    """Quick demo with another product."""
    
    print("\n🔄 Quick Pipeline Demo #2")
    print("=" * 30)
    
    runtime = InProcessRuntime()
    runtime.start()
    
    try:
        # Different product
        product = "Wireless earbuds with 30-hour battery, noise cancellation, and workout-proof design"
        
        print(f"📝 Product: {product}")
        
        # Simple pipeline without detailed logging
        simple_pipeline = SequentialOrchestration(members=marketing_agents)
        
        result = await simple_pipeline.invoke(
            task=product,
            runtime=runtime
        )
        
        final_copy = await result.get(timeout=30)
        print(f"\n🎯 Final Copy:\n{final_copy}")
        
    finally:
        await runtime.stop_when_idle()

# Run quick demo
await quick_pipeline_demo()

## Tóm tắt: Những gì bạn đã học

In [ ]:
print("🎓 Complete Tutorial Summary:")
print("=" * 35)

summary = [
    "✅ Basic AI agent with OpenAI",
    "✅ Custom tools/functions for weather and math", 
    "✅ Conversation memory management",
    "✅ Tool execution monitoring",
    "✅ Real-time streaming responses",
    "✅ Sequential agent orchestration"
]

for item in summary:
    print(item)

print("\n🚀 Key Concepts:")
concepts = {
    "Agent": "AI that can reason, remember, and use tools",
    "Kernel": "Manages AI services and functions",
    "Functions": "Tools that extend agent capabilities", 
    "Thread": "Maintains conversation history",
    "Streaming": "Real-time response generation",
    "Sequential Orchestration": "Pipeline where agents process output sequentially"
}

for concept, description in concepts.items():
    print(f"• {concept}: {description}")

print("\n💡 Agent Patterns:")
patterns = [
    "Single Agent: One agent handles entire workflow",
    "Sequential: Agents work in pipeline (A → B → C)",
    "Concurrent: Multiple agents work simultaneously", 
    "Manager: Central agent coordinates specialists",
    "Handoff: Agents pass control to each other"
]

for pattern in patterns:
    print(f"• {pattern}")

print("\n🎯 When to Use Sequential Orchestration:")
use_cases = [
    "• Document processing (extract → summarize → format)",
    "• Content creation (research → write → edit)", 
    "• Data analysis (collect → analyze → visualize)",
    "• Code review (analyze → suggest → validate)"
]

for use_case in use_cases:
    print(use_case)

**🔑 Những điểm chính cần lưu ý:**
- **Single Agents**: Tuyệt vời cho các tác vụ đơn giản và học hỏi
- **Sequential Orchestration**: Hoàn hảo cho các quy trình làm việc nhiều bước, trong đó mỗi bước được xây dựng dựa trên bước trước đó
- **Specialization**: Mỗi agent tập trung vào điểm mạnh nhất của mình
- **Pipeline Benefits**: Chất lượng tốt hơn thông qua xử lý chuyên biệt
- **Real-world Applications**: Xử lý tài liệu, tạo nội dung, phân tích dữ liệu

Hướng dẫn này bao gồm mọi thứ từ các agent cơ bản đến các multi-agent pipeline phức tạp!